# C4-classical-ml-practice — Practice p15 — Solution

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
X, y = load_wine(return_X_y=True)
ks = np.array([1, 3, 5, 7, 9, 11])
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
cv_results = []
for k in ks:
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=int(k))),
    ])
    cv_results.append(cross_val_score(candidate, X_tr, y_tr, cv=5).mean())
cv_means = np.array(cv_results, dtype=float)
best_k = int(ks[np.argmax(cv_means)])
best_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k)),
])
best_pipe.fit(X_tr, y_tr)
test_acc = float(best_pipe.score(X_te, y_te))

cv_means, best_k, test_acc

`np.argmax` returns the first maximum, so ascending `ks` implements the smallest-k tie rule without extra branching. The means for k = 7, 9, and 11 all round to 0.9544, so the rule selects 7, preserving more local detail instead of adding smoothing that produced no measured gain; its one-time test accuracy is 1.0.

### Answer check

In [ ]:
expected_cv = np.array([0.9327635327635327, 0.9396011396011396, 0.9396011396011396, 0.9544159544159545, 0.9544159544159545, 0.9544159544159545])
assert cv_means.shape == (6,) and np.allclose(cv_means, expected_cv)
assert np.unique(np.round(cv_means[3:], 4)).size == 1
assert best_k == 7 and isinstance(best_k, int)
assert np.isclose(test_acc, 1.0)